# S6E9 · 08：条件统计 + 残差特征 + 内部早停重训

**目标：尝试突破当前最高公榜 0.94618，冲击 0.947。0.947 是目标，不是本文件已实现的成绩。**

这不是 07 改个权重：新增一套带不同特征的 LightGBM 管线，旧 CatBoost、04 LightGBM 不重训。
程序不联网、不安装软件、不读取隐藏测试标签、不使用外部标签、不自动提交 Kaggle。

**现在怎么做：**使用能成功运行 04 的 Python 内核，确认第 1 节路径，然后 **Run All / 运行全部**。
Run All 会训练；没有 `False/True` 开关。每个外层折先做一次内部早停训练，再做一次固定轮数重训，共 10 次 LightGBM 拟合，另有小型主效应回归的内部交叉拟合。
这比只训练五个模型更耗时。第一折结束会打印实际耗时；机器速度和早停轮数未知，不能预先保证总耗时。

**保留目前 0.94618 的提交及原始文件，不覆盖。**本 notebook 不需要重新执行 03–07，但需要原始 `data` 和 04 的旧输出目录。

## 这次变更的理由（不需要修改）

**新信息，而不只是新权重。**保留原始特征、数字位和单列目标编码，新增收入与补贴、收入与环保关注程度等少量联合统计；同时告诉模型每项统计背后有多少样本。

**残差特征。**先在允许使用的训练子集内拟合一个小型 LogisticRegression 主效应模型，再统计 `真实标签 - 主效应预测` 在收入取值及联合组内的平均偏差。
它尝试区分“普遍由收入、补贴等因素解释的部分”和“同一取值仍剩下的偏差”。这是一项待验证假设，不是已经还原的合成数据生成公式。

**阻断明显的验证复用路径。**新模型的外层验证标签不参与目标编码、残差统计或早停。早停只能在外层训练数据内部决定；选好轮数后，重新用完整外层训练折拟合模型。
这些外层折和旧模型此前已参与过开发，因此依然不能把本次全部 OOF 称为全新的独立审计。

**不再把单模型排除在提交候选之外。**同时比较旧 LightGBM、旧 50/50、新 LightGBM，以及两种预先固定的融合；不用一长串权重试到公榜上升。

参考：[交叉拟合目标编码](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)、[区分模型选择与外层评估](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)、[LightGBM sklearn 接口](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html)。

## 1. 路径与运行设置

默认路径来自本会话的 04 输出。只有你移动过项目时才需要改。

`HISTORICAL_BEST_CSV` 可选：填入**当时实际提交并得到 0.94618 的 CSV 原文件**。程序会按 ID 和概率内容匹配，而不是猜测文件名。
暂时没找到就保留 `None`，仍然可以训练，但程序会注明最高分模型身份未核实，并保留两种旧对照。

不需要新装 XGBoost、CatBoost 或 kagglehub。使用已有 `numpy pandas scipy scikit-learn lightgbm joblib` 环境。
如确实缺依赖，再在当前内核单独运行 `%pip install lightgbm joblib`；不要为了运行 08 升级整套原环境。

In [1]:
from pathlib import Path

PROJECT = Path.cwd().parent  # 项目根目录（notebooks/ 的上一级）
V4_COMPARE_DIR = PROJECT / "outputs/v4_compare_20260930_163707_027699"
THREADS = 4

# 可选：确切的历史最高分 CSV；不能仅凭文件名确定它是哪种模型。
HISTORICAL_BEST_CSV = None
# 例如，你自己找到原文件后填写：
# HISTORICAL_BEST_CSV = PROJECT / "best_submission_094618.csv"


## 2. 定义函数（不训练）

下面是完整实现。无需打开、导入或运行旧 notebook。
LightGBM 深度 5、31 个叶节点、学习率 0.05、最多 5000 轮，沿用 04 的树模型设置；改变的是特征管线及选择训练轮数的方法。
新增联合统计的平滑值是预设实验值，不是已经证明的最优值。

In [2]:
from __future__ import annotations
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import time
import types
import warnings
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}

def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p

def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()

def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds

def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}

def save_json(path: Path, value: dict) -> None:
    """Atomic writes prevent incomplete metadata from being marked complete."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def check_source_files(bundle: dict) -> None:
    require(input_hashes(bundle["project"]) == bundle["hashes"],
            "原始 CSV 已改变，请停止并核对，不要混用不同版本的数据。")
    require(file_hash(bundle["compare_dir"] / "continuation_summary.json") == bundle["summary_sha256"],
            "04 的比较摘要已改变；请重新加载并核对来源。")

def load_v4(project: Path, compare_dir: Path,
            expected_oof: float | None = 0.94598624) -> dict:
    """Read ONLY the explicitly selected V4 run. Never guess the newest folder."""
    project, compare_dir = Path(project), Path(compare_dir)
    summary_file = compare_dir / "continuation_summary.json"
    if not summary_file.is_file():
        raise FileNotFoundError(f"找不到 {summary_file}。请核对 04 第 5 节打印的比较目录。")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    require(summary.get("selected_by_development_oof", {}).get("candidate") == "blend_cat50_lgb50",
            "这个目录不是截图中的 04 的 50/50 最优融合，不能悄悄换成别的运行。")
    train, test, y, folds = load_project(project)
    for frame in (train, test):
        for col in NUMERIC:
            require(col in frame, f"原始数据缺少必要列：{col}")
            require(np.isfinite(frame[col].to_numpy(dtype=float)).all(), f"{col} 包含非有限数值")
        require(np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round()), "收入不是预期整数")
        require(np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round()), "通勤距离不是预期一位小数")
    hashes = input_hashes(project)
    require(summary.get("input_sha256") == hashes,
            "当前数据哈希与 04 记录不一致。请恢复当时的数据，不要跳过检查。")
    cat_dir = Path(summary["baseline_dir"])
    lgb_dir = Path(summary["lightgbm_dir"])
    fingerprint = lgb_dir / "data_fingerprints.json"
    if not fingerprint.is_file():
        raise FileNotFoundError(f"找不到 {fingerprint}。请保留 04 的原 LightGBM 输出目录。")
    require(json.loads(fingerprint.read_text()) == hashes, "04 的 LightGBM 数据版本不一致。")
    cat = read_run(cat_dir, "catboost", train, test, y, folds)
    lgb4 = read_run(lgb_dir, "lightgbm", train, test, y, folds)
    base = {"oof": 0.5 * cat["oof"] + 0.5 * lgb4["oof"],
            "test": 0.5 * cat["test"] + 0.5 * lgb4["test"]}
    saved_oof = align_csv(compare_dir / "oof_blend_cat50_lgb50.csv", train.id,
                          ["target", "prediction"])
    saved_test = align_csv(compare_dir / "submission_next_candidate.csv", test.id, [TARGET])
    require(np.array_equal(saved_oof.target.to_numpy(), y.to_numpy()), "04 融合 OOF 标签不一致。")
    require(np.allclose(saved_oof.prediction, base["oof"], atol=1e-12, rtol=0),
            "04 的融合 OOF 不等于两个来源模型的 50/50。")
    require(np.allclose(saved_test[TARGET], base["test"], atol=1e-12, rtol=0),
            "04 的候选提交不等于来源模型的 50/50。")
    base["auc"] = float(roc_auc_score(y, base["oof"]))
    require(abs(base["auc"] - summary["selected_by_development_oof"]["oof_auc"]) < 1e-10,
            "04 摘要分数与保存的预测不一致。")
    if expected_oof is not None:
        require(abs(base["auc"] - expected_oof) < 1e-8,
                f"载入的融合 OOF {base['auc']:.10f} 与截图的 {expected_oof:.8f} 不一致。")
    bundle = {"project": project, "compare_dir": compare_dir, "cat_dir": cat_dir,
              "lgb04_dir": lgb_dir, "hashes": hashes, "summary_sha256": file_hash(summary_file),
              "train": train, "test": test, "y": y, "folds": folds,
              "base": base, "models": {"catboost": cat, "lgb04": lgb4}}
    check_source_files(bundle)
    print("04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。")
    print("CatBoost 来源：", cat_dir)
    print("LightGBM 来源：", lgb_dir)
    print(f"当前 04 融合 OOF：{base['auc']:.10f}")
    print("以上只核对了 04 本地结果；尚未证明哪个 CSV 对应公榜 0.94618。")
    return bundle

def submission_frame(bundle: dict, pred: np.ndarray) -> pd.DataFrame:
    test = bundle["test"]
    pred = probabilities(pred, len(test), "Submission")
    frame = pd.DataFrame({"id": test.id.to_numpy(), TARGET: pred})
    sample = bundle["project"] / "data/sample_submission.csv"
    if sample.is_file():
        ids = pd.read_csv(sample, usecols=["id"])
        frame = ids.merge(frame, on="id", how="left", validate="one_to_one", sort=False)
    probabilities(frame[TARGET], len(test), "Ordered submission")
    return frame

def runtime_code_fingerprint(*functions) -> str:
    """Ignore notebook execution filenames, but fingerprint executable logic."""
    def norm(value):
        if isinstance(value, types.CodeType):
            return {"bytecode": value.co_code.hex(), "constants": norm(value.co_consts),
                    "names": value.co_names, "vars": value.co_varnames,
                    "free": value.co_freevars, "cells": value.co_cellvars,
                    "argc": value.co_argcount, "posonly": value.co_posonlyargcount,
                    "kwonly": value.co_kwonlyargcount, "flags": value.co_flags,
                    "exception_table": getattr(value, "co_exceptiontable", b"").hex()}
        if isinstance(value, (tuple, list)):
            return [norm(v) for v in value]
        if isinstance(value, (set, frozenset)):
            return {"set": sorted([norm(v) for v in value], key=lambda v: json.dumps(v, sort_keys=True))}
        if isinstance(value, dict):
            return {str(k): norm(v) for k, v in value.items()}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)
    payload = [{"name": f.__name__, "code": norm(f.__code__),
                "defaults": norm(f.__defaults__), "kwdefaults": norm(f.__kwdefaults__)} for f in functions]
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

# 08 is one new feature pipeline, not a promise of leaderboard performance.
SPEC08 = {
    'n_estimators': 5000, 'learning_rate': 0.05,
    'max_depth': 5, 'num_leaves': 31, 'min_child_samples': 100, 'reg_lambda': 5.0,
    'early_stopping_rounds': 200, 'early_stop_fraction': 0.15,
    'inner_folds': 5, 'single_smoothing': 1.0, 'joint_smoothing': 20.0,
    'residual_smoothing': 20.0, 'anchor_max_rows': 150000,
    'anchor_C': 1.0, 'anchor_max_iter': 400,
}
REFERENCE08 = 'ref04_cat50_lgb50'
CANDIDATES08 = {
    REFERENCE08: {'cat': 0.50, 'lgb04': 0.50},
    'ref04_lgb_only': {'lgb04': 1.00},
    'v08_new_lgb_only': {'lgb08': 1.00},
    'v08_cat25_newlgb75': {'cat': 0.25, 'lgb08': 0.75},
    'v08_cat25_oldlgb25_newlgb50': {'cat': 0.25, 'lgb04': 0.25, 'lgb08': 0.50},
}


def versions08():
    try:
        import lightgbm
    except (ImportError, OSError) as e:
        raise RuntimeError('请使用之前运行 04 的内核。确实缺包时先运行 %pip install lightgbm。') from e
    return {k: importlib.metadata.version(k) for k in
            ['numpy','pandas','scipy','scikit-learn','lightgbm','joblib']}


def identify_best_submission(bundle, best_csv=None):
    """Match actual probabilities, NEVER infer leaderboard provenance from a name."""
    result={'reported_public_best':0.94618, 'status':'UNVERIFIED', 'matched_candidates':[]}
    if best_csv is None:
        print('最高分身份：未核实。程序将分别保留旧 LightGBM、旧 50/50 两个对照。')
        return result
    path=Path(best_csv).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f'指定的历史提交 CSV 不存在：{path}')
    uploaded=align_csv(path,bundle['test'].id,[TARGET])[TARGET].to_numpy()
    probabilities(uploaded,len(bundle['test']),'Historical submitted CSV')
    saved={'old_catboost':bundle['models']['catboost']['test'],
           'ref04_lgb_only':bundle['models']['lgb04']['test'],
           REFERENCE08:bundle['base']['test']}
    saved['old_cat25_lgb75']=0.25*saved['old_catboost']+0.75*saved['ref04_lgb_only']
    matches=[k for k,v in saved.items() if np.allclose(uploaded,v,atol=1e-12,rtol=0)]
    result.update(status='MATCHED' if len(matches)==1 else 'UNKNOWN_OR_AMBIGUOUS',
                  matched_candidates=matches, source_csv=str(path), sha256=file_hash(path),
                  max_abs_difference={k:float(np.max(np.abs(uploaded-v))) for k,v in saved.items()})
    print('你指定的最高分 CSV 内容匹配：', matches or '没有匹配到已知旧预测；不会按文件名猜测。')
    return result


def row_features08(frame):
    """Only row-local, label-free features. No automatic inference of target formulas."""
    x=frame.drop(columns=['id',TARGET],errors='ignore').copy().reset_index(drop=True)
    for col,scale,positions in [('Age',1,2),('Annual_Income_USD',1,6),('Daily_Commute_km',10,3),
                                 ('Charging_Stations_Near_Home',1,2),('Charging_Stations_Near_Work',1,2)]:
        v=np.rint(x[col].to_numpy()*scale).astype(np.int64)
        for j in range(positions):
            x[f'{col}_digit_{j}']=(v//10**j%10).astype('int8')
    income=np.rint(x['Annual_Income_USD']).astype('int64')
    x['income_last_2']=income%100
    x['income_last_3']=income%1000
    x['income_prefix_100']=income//100
    x['income_prefix_1000']=income//1000
    x['income_bucket_5000']=income//5000
    x['age_bucket_5']=np.floor(x['Age']/5).astype('int16')
    x['commute_bucket_10']=np.floor(x['Daily_Commute_km']/10).astype('int16')
    home=x['Charging_Stations_Near_Home']; work=x['Charging_Stations_Near_Work']
    x['charging_total']=home+work
    x['charging_min']=np.minimum(home,work)
    x['charging_difference']=work-home
    x['commute_per_station']=x['Daily_Commute_km']/(1+home+work)
    x['income_per_car']=income/(1+x['Number_of_Cars_Owned'])
    x['log_income']=np.log1p(np.maximum(income,0))
    for col in x.select_dtypes(include=['object','string','category']):
        x[col]=x[col].astype(str)
    for col in x.select_dtypes(include=['number']):
        require(np.isfinite(x[col]).all(),f'Nonfinite feature: {col}')
    return x


def anchor_features08(raw):
    """A small main-effects model; deliberately exclude IDs/digits/high-cardinality income categories."""
    z=raw.drop(columns=['id',TARGET],errors='ignore').copy().reset_index(drop=True)
    z['log_income_main']=np.log1p(np.maximum(z['Annual_Income_USD'],0))
    z['concern_squared_main']=z['Environmental_Concern_Level']**2
    for c in z.select_dtypes(include=['object','string','category']):
        z[c]=z[c].astype(str)
    return z


def fit_anchor08(raw, labels, seed, spec):
    z=anchor_features08(raw)
    cats=z.select_dtypes(include=['object','string','category']).columns.tolist()
    nums=[c for c in z if c not in cats]
    # Keep dependencies compatible with the user's existing environment.
    prep=ColumnTransformer([('numeric',StandardScaler(),nums),
                            ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=True),cats)],
                           sparse_threshold=1.0)
    model=make_pipeline(prep,LogisticRegression(C=spec['anchor_C'],solver='lbfgs',
                         max_iter=spec['anchor_max_iter'],tol=1e-5))
    idx=np.arange(len(z))
    if len(z)>spec['anchor_max_rows']:
        idx,_=train_test_split(idx,train_size=spec['anchor_max_rows'],
                               stratify=labels,random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',ConvergenceWarning)
        model.fit(z.iloc[idx],np.asarray(labels)[idx])
    if any(issubclass(w.category,ConvergenceWarning) for w in caught):
        print('提示：主效应 LogisticRegression 达到迭代上限；本次记录这一情况，未隐藏。',flush=True)
    return model


def group_specs08(x):
    cats=x.select_dtypes(include=['object','string','category']).columns.tolist()
    singles=list(dict.fromkeys(NUMERIC+cats+['income_prefix_100','income_prefix_1000']))
    groups=[(c,) for c in singles]
    income='Annual_Income_USD'; env='Environmental_Concern_Level'
    subsidy=[c for c in cats if 'subsid' in c.lower()]
    if len(subsidy)>1:
        raise ValueError(f'找到多个补贴列，不能静默选择：{subsidy}')
    pairs=[(income,env),('income_prefix_1000',env),('income_last_2',env),
           ('age_bucket_5',env),('Charging_Stations_Near_Home','Charging_Stations_Near_Work')]
    if subsidy:
        s=subsidy[0]
        pairs += [(income,s),('income_prefix_1000',s),('income_last_2',s),(s,env)]
    return groups+pairs


def lookup08(x, columns, table):
    if len(columns)==1:
        key=pd.Index(x[columns[0]].to_numpy(),name=columns[0])
    else:
        key=pd.MultiIndex.from_frame(x[list(columns)])
    return table.reindex(key)


class FeatureEncoder08:
    """Fit on an allowed training subset; transform never accepts labels."""
    def __init__(self,spec,seed):
        self.spec=dict(spec); self.seed=int(seed)

    def fit(self,raw,labels):
        raw=raw.reset_index(drop=True)
        y=np.asarray(labels,dtype=float)
        require(len(raw)==len(y) and set(np.unique(y))=={0.,1.},'编码训练标签非法。')
        x=row_features08(raw)
        self.groups_=group_specs08(x)
        self.prior_=float(y.mean())
        self.anchor_=fit_anchor08(raw,y,self.seed,self.spec)
        p=self.anchor_.predict_proba(anchor_features08(raw))[:,1]
        self.tables_=[]
        for columns in self.groups_:
            table=x[list(columns)].copy()
            table['_label']=y
            table['_residual']=y-p
            stats=table.groupby(list(columns),observed=True,dropna=False,sort=False).agg(
                positive=('_label','sum'),count=('_label','size'),residual=('_residual','sum'))
            self.tables_.append(stats)
        return self

    def encoded_names(self):
        return ['main_effect_probability'] + [f'group_{i:02d}_{s}' for i in range(len(self.groups_))
                                                for s in ['te','log_count','residual']]

    def transform(self,raw):
        x=row_features08(raw)
        p=self.anchor_.predict_proba(anchor_features08(raw))[:,1]
        matrix=np.empty((len(x),1+3*len(self.groups_)),dtype='float32')
        matrix[:,0]=p
        for i,(columns,table) in enumerate(zip(self.groups_,self.tables_)):
            found=lookup08(x,columns,table).fillna(0.)
            count=found['count'].to_numpy()
            alpha=self.spec['single_smoothing'] if len(columns)==1 else self.spec['joint_smoothing']
            matrix[:,1+3*i]=(found.positive.to_numpy()+alpha*self.prior_)/(count+alpha)
            matrix[:,2+3*i]=np.log1p(count)
            matrix[:,3+3*i]=found.residual.to_numpy()/(count+self.spec['residual_smoothing'])
        require(np.isfinite(matrix).all(),'编码中出现 NaN / Infinity')
        return pd.DataFrame(matrix,columns=self.encoded_names())


def encoded_split08(raw_train,y_train,queries,seed,spec):
    """Cross-fit training rows. Query datasets are never used for fitting.

    KFold here intentionally ignores labels when defining the internal encoding splits;
    outer splits and early-stopping split are stratified separately.
    """
    rt=raw_train.reset_index(drop=True)
    labels=np.asarray(y_train)
    names=None; matrix=None
    kf=KFold(spec['inner_folds'],shuffle=True,random_state=seed)
    for j,(fit_idx,hold_idx) in enumerate(kf.split(rt),1):
        encoder=FeatureEncoder08(spec,seed+j).fit(rt.iloc[fit_idx],labels[fit_idx])
        transformed=encoder.transform(rt.iloc[hold_idx])
        if matrix is None:
            names=transformed.columns.tolist()
            matrix=np.full((len(rt),len(names)),np.nan,dtype='float32')
        require(names==transformed.columns.tolist(),'内层特征名称不同')
        matrix[hold_idx]=transformed.to_numpy()
        del encoder,transformed
    require(np.isfinite(matrix).all(),'训练编码覆盖不完整')
    full_encoder=FeatureEncoder08(spec,seed+100).fit(rt,labels)
    xtrain=pd.concat([row_features08(rt),pd.DataFrame(matrix,columns=names)],axis=1)
    result=[]
    for q in queries:
        q=q.reset_index(drop=True)
        result.append(pd.concat([row_features08(q),full_encoder.transform(q)],axis=1))
    cats=xtrain.select_dtypes(include=['object','string','category']).columns.tolist()
    categories={col:pd.unique(xtrain[col]).tolist() for col in cats}
    for frame in [xtrain]+result:
        require(frame.columns.tolist()==xtrain.columns.tolist(),'训练、查询特征不一致')
        for col in cats:
            frame[col]=frame[col].astype(pd.CategoricalDtype(categories=categories[col]))
        numeric=[c for c in frame if c not in cats]
        frame[numeric]=frame[numeric].astype('float32')
    full_encoder.categories_=categories
    full_encoder.feature_columns_=xtrain.columns.tolist()
    return xtrain,result,full_encoder


def inference_frame08(raw,encoder):
    frame=pd.concat([row_features08(raw),encoder.transform(raw)],axis=1)
    for col,values in encoder.categories_.items():
        frame[col]=frame[col].astype(pd.CategoricalDtype(categories=values))
    for col in frame:
        if col not in encoder.categories_:
            frame[col]=frame[col].astype('float32')
    require(frame.columns.tolist()==encoder.feature_columns_,'推理特征模式与训练不同')
    return frame


def lgb_model08(spec,threads,seed,n_estimators=None):
    import lightgbm as lgb
    params={k:spec[k] for k in ['learning_rate','max_depth','num_leaves','min_child_samples','reg_lambda']}
    return lgb.LGBMClassifier(**params,n_estimators=spec['n_estimators'] if n_estimators is None else n_estimators,
                            objective='binary',metric='auc',max_bin=255,
                            random_state=seed,n_jobs=threads,deterministic=True,force_col_wise=True,verbosity=-1)


@contextmanager
def lock08(directory):
    import fcntl
    with (directory/'.lock').open('a+') as f:
        try:
            fcntl.flock(f.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError as e:
            raise RuntimeError('另一个内核正在运行这套 08；不要同时运行两份。') from e
        try: yield
        finally: fcntl.flock(f.fileno(),fcntl.LOCK_UN)


def paths08(out,fold):
    return {'model':out/f'lightgbm_fold{fold}.txt',
            'encoder':out/f'encoder_fold{fold}.joblib',
            'prediction':out/f'lightgbm_fold{fold}_predictions.npz',
            'splits':out/f'splits_fold{fold}.npz',
            'history':out/f'pilot_history_fold{fold}.json',
            'importance':out/f'feature_importance_fold{fold}.csv'}


def read_fold08(out,fold,signature,train,test,ti,vi):
    paths=paths08(out,fold)
    marker=json.loads((out/f'fold{fold}_complete.json').read_text())
    require(marker.get('signature')==signature and marker.get('fold')==fold,'缓存配置标记不一致')
    require(set(marker['hashes'])=={p.name for p in paths.values()},'缓存标记缺少必要文件')
    for filename,checksum in marker['hashes'].items():
        p=out/filename
        require(p.is_file() and file_hash(p)==checksum,f'缓存文件缺失/改变：{p}')
    with np.load(paths['splits'],allow_pickle=False) as s:
        require(np.array_equal(s['outer_train'],ti) and np.array_equal(s['outer_valid'],vi),'缓存外层划分不一致')
        require(not np.intersect1d(s['pilot_train'],s['pilot_stop']).size,'早停划分重叠')
        require(np.array_equal(np.sort(np.r_[s['pilot_train'],s['pilot_stop']]),np.sort(ti)),'早停未局限在外层训练中')
    with np.load(paths['prediction'],allow_pickle=False) as p:
        require(np.array_equal(p['valid_indices'],vi),'缓存验证索引不一致')
        require(np.array_equal(p['valid_ids'],train.id.iloc[vi]),'缓存验证 ID 不一致')
        require(np.array_equal(p['test_ids'],test.id),'缓存测试 ID 不一致')
        vp=probabilities(p['valid_prediction'],len(vi),'Cached OOF')
        tp=probabilities(p['test_prediction'],len(test),'Cached test')
    return vp,tp,int(marker['rounds'])


def train08(bundle,threads=4,spec=None):
    import lightgbm as lgb
    spec=dict(SPEC08 if spec is None else spec)
    require(threads>=1,'THREADS 必须是正整数')
    require(set(spec)==set(SPEC08),'配置字段不完整')
    require(spec['inner_folds']>=2 and 0<spec['early_stop_fraction']<0.5,'编码/早停参数非法')
    check_source_files(bundle)
    versions=versions08()
    old_versions_file=bundle['lgb04_dir']/'package_versions.json'
    if old_versions_file.exists():
        old=json.loads(old_versions_file.read_text())
        print('依赖版本是否与 04 一致：',all(versions.get(k)==v for k,v in old.items() if k in versions))
    # A changed feature definition or trainer creates a different cache directory.
    funcs=[row_features08,anchor_features08,fit_anchor08,group_specs08,lookup08,
           FeatureEncoder08.fit,FeatureEncoder08.transform,FeatureEncoder08.encoded_names,
           encoded_split08,lgb_model08,train08,read_fold08]
    config={'revision':'v08-crossfit-residual-conditional-innerstop-refit-1',
            'models':['lightgbm'],'folds':5,'seed':42,
            'train_rows':len(bundle['train']),'test_rows':len(bundle['test']),
            'input_sha256':bundle['hashes'],'v4_summary_sha256':bundle['summary_sha256'],
            'spec':spec,'threads':threads,'versions':versions,'python':sys.version.split()[0],
            'platform':platform.platform(),'code_sha256':runtime_code_fingerprint(*funcs)}
    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    out=bundle['project']/'outputs'/f'v8_feature_lgb_{signature}'
    out.mkdir(parents=True,exist_ok=True)
    with lock08(out),threadpool_limits(limits=threads):
        cf=out/'config.json'
        if cf.exists():
            require(json.loads(cf.read_text())==config,'已有目录配置不同，停止复用')
        else:
            require(not any(p.name!='.lock' for p in out.iterdir()),'训练目录有未标记内容，停止')
            save_json(cf,config)
        print('08 训练目录：',out,flush=True)
        print('每折先用内部早停集选轮数，再在整个外层训练折重训。不会训练旧 CatBoost。',flush=True)
        train,test,y,folds=(bundle[k] for k in ['train','test','y','folds'])
        oof=np.full(len(y),np.nan); tsum=np.zeros(len(test)); records=[]
        groups=group_specs08(row_features08(train.iloc[:1]))
        save_json(out/'feature_groups.json',{'groups':[list(g) for g in groups],
                  'note':'单列TE、支持度、主效应残差和有限联合统计；不是已证实的生成公式'})
        print('联合统计键：',[g for g in groups if len(g)>1],flush=True)
        for fold,(ti,vi) in enumerate(folds,1):
            start=time.monotonic(); paths=paths08(out,fold)
            if (out/f'fold{fold}_complete.json').is_file():
                vp,tp,rounds=read_fold08(out,fold,signature,train,test,ti,vi)
                print(f'复用完整 Fold {fold}/5，rounds={rounds}',flush=True)
            else:
                print(f'\n===== Fold {fold}/5：内部早停 =====',flush=True)
                pilot,stop=train_test_split(ti,test_size=spec['early_stop_fraction'],
                                           stratify=y.iloc[ti],random_state=8000+fold)
                require(not np.intersect1d(vi,np.r_[pilot,stop]).size,'外层验证行进入了早停过程')
                a,query,_=encoded_split08(train.iloc[pilot],y.iloc[pilot],[train.iloc[stop]],9000+fold,spec)
                b=query[0]
                model=lgb_model08(spec,threads,41+fold)
                bins=[1023 if c=='Annual_Income_USD' else 255 for c in a.columns]
                model.set_params(max_bin_by_feature=bins)
                model.fit(a,y.iloc[pilot],eval_set=[(b,y.iloc[stop])],eval_metric='auc',
                          callbacks=[lgb.early_stopping(spec['early_stopping_rounds'],first_metric_only=True),
                                     lgb.log_evaluation(500)])
                rounds=int(model.best_iteration_ or spec['n_estimators'])
                save_json(paths['history'],model.evals_result_)
                del a,b,query,model,_;gc.collect()
                print(f'Fold {fold}: 内部早停确定 {rounds} 轮；现在重新编码整个外层训练折。',flush=True)
                a,query,encoder=encoded_split08(train.iloc[ti],y.iloc[ti],[train.iloc[vi],test],10000+fold,spec)
                b,c=query
                final=lgb_model08(spec,threads,41+fold,n_estimators=rounds)
                final.set_params(max_bin_by_feature=[1023 if col=='Annual_Income_USD' else 255 for col in a.columns])
                # Critical: NO eval_set and NO early-stopping labels from outer validation.
                final.fit(a,y.iloc[ti])
                require(np.array_equal(final.classes_,[0,1]),'正类概率顺序不一致')
                vp=probabilities(final.predict_proba(b)[:,1],len(vi),'New OOF')
                tp=probabilities(final.predict_proba(c)[:,1],len(test),'New test')
                final.booster_.save_model(str(paths['model']))
                joblib.dump(encoder,paths['encoder'],compress=3)
                np.savez_compressed(paths['splits'],outer_train=ti,outer_valid=vi,pilot_train=pilot,pilot_stop=stop)
                np.savez_compressed(paths['prediction'],valid_indices=vi,valid_ids=train.id.iloc[vi].to_numpy(),
                                    valid_prediction=vp,test_ids=test.id.to_numpy(),test_prediction=tp)
                pd.DataFrame({'feature':a.columns,'gain':final.booster_.feature_importance(importance_type='gain')}).sort_values(
                    'gain',ascending=False).to_csv(paths['importance'],index=False)
                # Ensure the saved booster and encoder reconstruct the same inference pipeline.
                probe=test.iloc[:min(128,len(test))]
                reloaded=lgb.Booster(model_file=str(paths['model']))
                check_pred=reloaded.predict(inference_frame08(probe,encoder))
                require(np.allclose(check_pred,tp[:len(probe)],rtol=0,atol=1e-10),'重载模型预测不一致')
                check_source_files(bundle)
                save_json(out/f'fold{fold}_complete.json',{'signature':signature,'fold':fold,'rounds':rounds,
                           'hashes':{p.name:file_hash(p) for p in paths.values()}})
                del a,b,c,query,encoder,final,reloaded;gc.collect()
            oof[vi]=vp;tsum+=tp
            auc=float(roc_auc_score(y.iloc[vi],vp))
            records.append({'fold':fold,'auc':auc,'rounds':rounds,'elapsed_seconds_this_run':time.monotonic()-start})
            pd.DataFrame(records).to_csv(out/'fold_scores.csv',index=False)
            print(f'Fold {fold}/5 外层 OOF AUC={auc:.8f}，本次用时={records[-1]["elapsed_seconds_this_run"]:.1f}s',flush=True)
        probabilities(oof,len(y),'Completed OOF')
        testpred=tsum/len(folds)
        pd.DataFrame({'id':train.id,'target':y,'prediction':oof}).to_csv(out/'oof_lightgbm.csv',index=False)
        pd.DataFrame({'id':test.id,TARGET:testpred}).to_csv(out/'submission_lightgbm.csv',index=False)
        result={'oof':oof,'test':testpred,'auc':float(roc_auc_score(y,oof))}
        save_json(out/'training_summary.json',{'oof_auc':result['auc'],'spec':spec,
                   'outer_validation_used_for_early_stopping':False,
                   'note':'Outer folds are previously reused development data, NOT a fresh independent audit.'})
        print('08 新模型五折结束；开发 OOF：',result['auc'],flush=True)
    return out,result


def auc_influence08(y,p):
    """Per-positive / per-negative AUC placements, with half-credit ties."""
    y=np.asarray(y);p=np.asarray(p,dtype=float)
    pos=p[y==1];neg=p[y==0]
    require(len(pos)>1 and len(neg)>1,'AUC 区间需要两个类别各至少两行')
    allrank=rankdata(np.r_[pos,neg],method='average')
    rp=rankdata(pos,method='average');rn=rankdata(neg,method='average')
    vpos=(allrank[:len(pos)]-rp)/len(neg)
    vneg=1-(allrank[len(pos):]-rn)/len(pos)
    return vpos,vneg


def conditional_delta_interval08(y,new,reference):
    """Paired placement-based SE, conditional on the saved predictions.

    NOT a CV-adjusted significance test: ignores training overlap and adaptive selection.
    """
    ap,an=auc_influence08(y,new);bp,bn=auc_influence08(y,reference)
    dp,dn=ap-bp,an-bn
    delta=float(dp.mean());se=float(np.sqrt(dp.var(ddof=1)/len(dp)+dn.var(ddof=1)/len(dn)))
    return delta-1.96*se,delta+1.96*se,se


def compare08(bundle,new_dir,new,identity):
    check_source_files(bundle)
    train,test,y,folds=(bundle[k] for k in ['train','test','y','folds'])
    config=json.loads((new_dir/'config.json').read_text())
    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
    reconstructed=np.full(len(y),np.nan);test_sum=np.zeros(len(test))
    for f,(ti,vi) in enumerate(folds,1):
        vp,tp,_=read_fold08(new_dir,f,signature,train,test,ti,vi)
        reconstructed[vi]=vp;test_sum+=tp
    require(np.allclose(reconstructed,new['oof'],atol=1e-12,rtol=0),'内存OOF与磁盘不符')
    require(np.allclose(test_sum/5,new['test'],atol=1e-12,rtol=0),'内存测试预测与磁盘不符')
    models={'cat':bundle['models']['catboost'],'lgb04':bundle['models']['lgb04'],'lgb08':new}
    pred={name:{part:sum(w*models[m][part] for m,w in weights.items()) for part in ['oof','test']}
          for name,weights in CANDIDATES08.items()}
    reference=pred[REFERENCE08]
    rows=[];byfold=[]
    base_f=[roc_auc_score(y.iloc[v],reference['oof'][v]) for _,v in folds]
    for name,pr in pred.items():
        score=float(roc_auc_score(y,pr['oof']))
        ds=[]
        for f,(_,vi) in enumerate(folds,1):
            fa=float(roc_auc_score(y.iloc[vi],pr['oof'][vi]));d=fa-base_f[f-1];ds.append(d)
            byfold.append({'candidate':name,'fold':f,'auc':fa,'delta_vs_old_blend':d})
        lo,hi,se=conditional_delta_interval08(y,pr['oof'],reference['oof'])
        rows.append({'candidate':name,'oof_auc':score,'delta_vs_old_blend':score-bundle['base']['auc'],
                     'improved_folds':sum(d>0 for d in ds),'worst_fold_delta':min(ds),
                     'conditional_delta_low95':lo,'conditional_delta_high95':hi})
    table=pd.DataFrame(rows).sort_values('oof_auc',ascending=False,kind='stable').reset_index(drop=True)
    folds_table=pd.DataFrame(byfold)
    oldrows=table[table.candidate.isin([REFERENCE08,'ref04_lgb_only'])]
    local_best_old=oldrows.iloc[0]
    best=table.iloc[0]
    # Do NOT gate selection on an invented fixed +0.00005 "significance" threshold.
    # This is only a conservative review filter; the conditional interval is not independent evidence.
    isnew=str(best.candidate).startswith('v08_')
    eligible=bool(isnew and best.oof_auc>local_best_old.oof_auc and best.improved_folds>=4
                  and best.conditional_delta_low95>0)
    status='REVIEW_NOT_PROVEN' if eligible else ('WEAK_LOCAL_SIGNAL' if isnew else 'KEEP_OLD')
    out=bundle['project']/'outputs'/('v8_compare_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True,exist_ok=False)
    table.to_csv(out/'comparison.csv',index=False)
    folds_table.to_csv(out/'comparison_by_fold.csv',index=False)
    ids={'train_ids':train.id.to_numpy(),'target':y.to_numpy(),'test_ids':test.id.to_numpy()}
    for name,p in pred.items():
        ids[name+'_oof']=p['oof'];ids[name+'_test']=p['test']
    np.savez_compressed(out/'predictions_for_review.npz',**ids)
    candidate_csv=None
    if eligible:
        candidate_csv=out/'submission_v08_review_candidate.csv'
        submission_frame(bundle,pred[best.candidate]['test']).to_csv(candidate_csv,index=False)
    report={'status':status,'raw_best_by_development_oof':best.to_dict(),
            'local_best_old':local_best_old.to_dict(),'candidate_specs':CANDIDATES08,
            'candidate_csv':str(candidate_csv) if candidate_csv else None,
            'reported_public_best_identity':identity,'v4_compare_dir':str(bundle['compare_dir']),
            'new_model_dir':str(new_dir),'input_sha256':bundle['hashes'],
            'new_public_score':None,'requested_goal_public_auc':0.947,
            'note':'0.947 is a goal, NOT a trained result. No external labels or online data used.',
            'interval_warning':'Conditional on saved predictions only; not CV-adjusted or corrected for adaptive selection. Review filter is not evidence of guaranteed leaderboard gain.',
            'validation_warning':'The new booster has inner early stopping and outer refit. Old predictions and these folds have already been used for model selection. NOT a new independent holdout.'}
    save_json(out/'summary.json',report)
    print('\n【08 主比较】')
    print(table[['candidate','oof_auc','delta_vs_old_blend','improved_folds','worst_fold_delta']].to_string(
        index=False,float_format=lambda a:f'{a:.8f}'))
    print('\n输出目录：',out)
    print('最高分身份：',identity['status'],'。不要仅凭文件名把 0.94618 对应到某个模型。')
    print('本地最高候选：',best.candidate,'，OOF：',f'{best.oof_auc:.8f}')
    print('状态：',status)
    if candidate_csv:
        print('唯一待评估 CSV：',candidate_csv,'\n先发主表截图；这不是保证达到 0.947 的文件。')
    else:
        print('未导出新提交 CSV；保留旧提交。本轮结果仍已保存，不需重复训练。')
    print('条件区间只衡量这些固定预测的样本层面变化，未计入训练依赖和反复选择，不能当作独立显著性证据。')
    return out,table,folds_table,report


## 3. 核对旧结果与最高分身份（不训练）

这里同时检查原始数据哈希、ID、标签、每折划分、预测汇总。最高分身份无法核实时不会编造对应关系，也不阻止新实验。

若报错找不到目录，请核对第 1 节路径；不要删除保护检查。

In [3]:
versions = versions08()
bundle = load_v4(PROJECT, V4_COMPARE_DIR)
identity = identify_best_submission(bundle, HISTORICAL_BEST_CSV)
print("当前 Python：", sys.executable)
print("当前依赖：", versions)
print("旧 LightGBM 单模型 OOF：", bundle["models"]["lgb04"]["auc"])
print("旧 CatBoost/LightGBM 50/50 OOF：", bundle["base"]["auc"])
print("下面第 4 节会启动新模型训练；历史 0.94618 仍保留。")


04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。
CatBoost 来源： /Users/yw/unsw/Kaggle/outputs/v3_20260925_180457_278867
LightGBM 来源： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
当前 04 融合 OOF：0.9459862434
以上只核对了 04 本地结果；尚未证明哪个 CSV 对应公榜 0.94618。
最高分身份：未核实。程序将分别保留旧 LightGBM、旧 50/50 两个对照。
当前 Python： /opt/miniconda3/envs/dl/bin/python
当前依赖： {'numpy': '2.4.6', 'pandas': '3.0.6', 'scipy': '1.17.1', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'joblib': '1.6.0'}
旧 LightGBM 单模型 OOF： 0.9459463810009869
旧 CatBoost/LightGBM 50/50 OOF： 0.945986243390633
下面第 4 节会启动新模型训练；历史 0.94618 仍保留。


## 4. 训练 08（运行这一格就会开始）

流程：外层训练折 → 内部拟合/早停划分 → 在内部拟合数据中交叉拟合特征 → 选轮数 → 在整个外层训练折重新编码并固定轮数重训 → 预测外层验证和测试。

只保留一个新模型配置，不做自动网格搜索。完整结束的折会校验后复用；中断的一折从头训练。
重启内核后 Run All 可复用相同配置下完整保存的折。不要同时在两个内核运行这套实验。

每折保存模型、编码器、训练索引、内部早停索引、预测、特征重要性和完成标记。编码器为本程序自己保存的 joblib 文件；不要加载陌生来源的 joblib/pickle。

In [4]:
lgb08_dir, lgb08_result = train08(bundle, threads=THREADS)
print("08 的五折已完成。继续第 5 节查看比较。")


依赖版本是否与 04 一致： True
08 训练目录： /Users/yw/unsw/Kaggle/outputs/v8_feature_lgb_08b876dc6cfb761a
每折先用内部早停集选轮数，再在整个外层训练折重训。不会训练旧 CatBoost。
联合统计键： [('Annual_Income_USD', 'Environmental_Concern_Level'), ('income_prefix_1000', 'Environmental_Concern_Level'), ('income_last_2', 'Environmental_Concern_Level'), ('age_bucket_5', 'Environmental_Concern_Level'), ('Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'), ('Annual_Income_USD', 'Subsidy_Available'), ('income_prefix_1000', 'Subsidy_Available'), ('income_last_2', 'Subsidy_Available'), ('Subsidy_Available', 'Environmental_Concern_Level')]

===== Fold 1/5：内部早停 =====


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[239]	valid_0's auc: 0.945089
Evaluated only: auc
Fold 1: 内部早停确定 239 轮；现在重新编码整个外层训练折。
Fold 1/5 外层 OOF AUC=0.94468946，本次用时=41.1s

===== Fold 2/5：内部早停 =====


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[250]	valid_0's auc: 0.946831
Evaluated only: auc
Fold 2: 内部早停确定 250 轮；现在重新编码整个外层训练折。
Fold 2/5 外层 OOF AUC=0.94558452，本次用时=42.1s

===== Fold 3/5：内部早停 =====


...

Training until validation scores don't improve for 200 rounds
[500]	valid_0's auc: 0.946722
Early stopping, best iteration is:
[347]	valid_0's auc: 0.946987
Evaluated only: auc
Fold 3: 内部早停确定 347 轮；现在重新编码整个外层训练折。
Fold 3/5 外层 OOF AUC=0.94647754，本次用时=48.3s

===== Fold 4/5：内部早停 =====


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[179]	valid_0's auc: 0.945938
Evaluated only: auc
Fold 4: 内部早停确定 179 轮；现在重新编码整个外层训练折。
Fold 4/5 外层 OOF AUC=0.94592084，本次用时=39.0s

===== Fold 5/5：内部早停 =====


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[500]	valid_0's auc: 0.945715
Early stopping, best iteration is:
[303]	valid_0's auc: 0.945854
Evaluated only: auc
Fold 5: 内部早停确定 303 轮；现在重新编码整个外层训练折。
Fold 5/5 外层 OOF AUC=0.94587497，本次用时=45.5s
08 新模型五折结束；开发 OOF： 0.9456826224803709
08 的五折已完成。继续第 5 节查看比较。


## 5. 比较五个方案

| 候选 | 含义 |
|---|---|
| `ref04_cat50_lgb50` | 旧 CatBoost / 旧 LightGBM = 50/50 |
| `ref04_lgb_only` | 旧 LightGBM 单模型 |
| `v08_new_lgb_only` | 新特征 LightGBM 单模型 |
| `v08_cat25_newlgb75` | 25% 旧 CatBoost + 75% 新 LightGBM |
| `v08_cat25_oldlgb25_newlgb50` | 25% 旧 CatBoost + 25% 旧 LightGBM + 50% 新 LightGBM |

`delta_vs_old_blend` 统一相对**旧 50/50 的本地 OOF**，不是相对公榜。
新增“固定预测条件下的配对 AUC 差值区间”，它不计入交叉验证训练依赖、此前调参和多次选择，**不是独立显著性证明，也不是公榜分数区间**。

仅当新候选是开发 OOF 最高、超过两个旧对照、至少四折相对旧融合提高且条件区间下端大于零时，导出一个待评估 CSV。
这仍只是保守的人工复核筛选规则，不能保证提升，更不能保证达到 0.947。

In [5]:
comparison_dir, comparison, by_fold, result_summary = compare08(
    bundle, lgb08_dir, lgb08_result, identity
)
from IPython.display import display
main_cols = ["candidate", "oof_auc", "delta_vs_old_blend", "improved_folds", "worst_fold_delta"]
print("主比较表：请先把这张表截图发来。")
display(comparison[main_cols].style.format({
    "oof_auc": "{:.8f}", "delta_vs_old_blend": "{:+.8f}", "worst_fold_delta": "{:+.8f}"
}))
print("逐折变化（相对旧 50/50）：")
display(by_fold.pivot(index="fold", columns="candidate", values="delta_vs_old_blend")
        .style.format("{:+.8f}"))
print("状态：", result_summary["status"])
print("已知最高公榜 0.94618 是历史记录；本次还没有新的 Kaggle Score。")



【08 主比较】
                  candidate    oof_auc  delta_vs_old_blend  improved_folds  worst_fold_delta
v08_cat25_oldlgb25_newlgb50 0.94599817          0.00001193               2       -0.00004918
          ref04_cat50_lgb50 0.94598624          0.00000000               0        0.00000000
             ref04_lgb_only 0.94594638         -0.00003986               1       -0.00009085
         v08_cat25_newlgb75 0.94584620         -0.00014005               1       -0.00021232
           v08_new_lgb_only 0.94568262         -0.00030362               0       -0.00042156

输出目录： /Users/yw/unsw/Kaggle/outputs/v8_compare_20260930_201751_588159
最高分身份： UNVERIFIED 。不要仅凭文件名把 0.94618 对应到某个模型。
本地最高候选： v08_cat25_oldlgb25_newlgb50 ，OOF： 0.94599817
状态： WEAK_LOCAL_SIGNAL
未导出新提交 CSV；保留旧提交。本轮结果仍已保存，不需重复训练。
条件区间只衡量这些固定预测的样本层面变化，未计入训练依赖和反复选择，不能当作独立显著性证据。
主比较表：请先把这张表截图发来。


,candidate,oof_auc,delta_vs_old_blend,improved_folds,worst_fold_delta
0,v08_cat25_oldlgb25_newlgb50,0.94599817,+0.00001193,2,-0.00004918
1,ref04_cat50_lgb50,0.94598624,+0.00000000,0,+0.00000000
2,ref04_lgb_only,0.94594638,-0.00003986,1,-0.00009085
3,v08_cat25_newlgb75,0.94584620,-0.00014005,1,-0.00021232
4,v08_new_lgb_only,0.94568262,-0.00030362,0,-0.00042156


逐折变化（相对旧 50/50）：


candidate,ref04_cat50_lgb50,ref04_lgb_only,v08_cat25_newlgb75,v08_cat25_oldlgb25_newlgb50,v08_new_lgb_only
fold,,,,,
1,+0.00000000,-0.00003641,-0.00018406,-0.00002724,-0.00034703
2,+0.00000000,-0.00009085,+0.00002506,+0.00012224,-0.00009644
3,+0.00000000,-0.00007242,-0.00021115,-0.00004918,-0.00042156
4,+0.00000000,-0.00000344,-0.00021232,-0.00002755,-0.00038566
5,+0.00000000,+0.00000562,-0.00005255,+0.00007387,-0.00016436


状态： WEAK_LOCAL_SIGNAL
已知最高公榜 0.94618 是历史记录；本次还没有新的 Kaggle Score。


## 输出与下一步

**先发第 5 节主表截图，不用立即向 Kaggle 提交。**

`KEEP_OLD`：旧方案本地仍更好；保留最高分历史提交，没有新候选 CSV。

`WEAK_LOCAL_SIGNAL`：新方案局部有改善，但未通过本次复核筛选；预测已保存供分析，没有新候选 CSV。

`REVIEW_NOT_PROVEN`：有一份值得人工评估的候选，程序打印唯一文件的完整路径：`submission_v08_review_candidate.csv`。
它还不是保证提高公榜的文件，不能据此删除历史最佳提交。

训练目录：`outputs/v8_feature_lgb_<指纹>/`。比较目录：`outputs/v8_compare_<时间戳>/`。
不要提交训练目录里的 `submission_lightgbm.csv`，除非人工复核明确决定测试这个单模型；日常操作只看比较目录打印的唯一候选。

后续分析需要的 OOF、测试概率、ID、标签、配置和分数，保存在比较目录的 `predictions_for_review.npz`、`comparison.csv`、`comparison_by_fold.csv`、`summary.json` 中。

**验证情况：**已通过 Notebook 结构与语法检查，以及小型合成数据流程测试（700 行训练、140 行测试，测试时将轮数上限降到 24）：完整五折内部早停与外层重训、编码内层留出标签不影响其自身特征、按概率匹配历史提交、模型和编码器保存/重载、未知类别、完整折复用、损坏缓存拒绝、提交行顺序及导出/保留旧版分支。
这些测试验证程序功能，**不是竞赛效果验证**。当前没有用户真实竞赛 CSV 和实际最高分 CSV，本版没有真实新增 OOF 或公榜成绩。